# Uncapacitated Facility Location problem

## Gurobi setup and academic license

Open the course repository root in VS Code and select the Julia 1.12 kernel with the course environment. Run **ISyE 524: Update course repository** to install the included Gurobi Julia package. If Julia reports that `Gurobi` is missing, run this **once in the Julia REPL with the course project active**, then restart the notebook kernel:

```julia
import Pkg
Pkg.add("Gurobi"; preserve = Pkg.PRESERVE_ALL)
```

The code below assumes Gurobi is installed and your license is in a default location. No installation paths need to be set in the notebook.

To obtain a free academic license:

1. Create a Gurobi account with your university email and sign in to the [Gurobi User Portal](https://portal.gurobi.com/).
2. While on the university network (or a VPN that routes through it), request an **Academic Named-User** license for the computer on which Julia runs.
3. Copy the portal's `grbgetkey YOUR-LICENSE-KEY` command into a terminal on that computer. With a standard Gurobi installation, the utility is in its `bin` directory if it is not on your PATH. Accept the default location for `gurobi.lic`, or save it in your home directory. In WSL, activate the license inside WSL.
4. Restart the Julia kernel and choose **Run All**. The first code cell solves a tiny LP to check access before building the larger facility-location model. Keep the license file outside this repository.

See [Gurobi's academic-license instructions](https://support.gurobi.com/hc/en-us/articles/360040541251-How-do-I-obtain-a-free-academic-license), [license activation](https://support.gurobi.com/hc/en-us/articles/13207658935185-How-do-I-retrieve-an-Academic-Named-User-license), [default license locations](https://support.gurobi.com/hc/en-us/articles/360013417211-Where-do-I-place-the-Gurobi-license-file-gurobi-lic), and [Gurobi.jl setup](https://jump.dev/JuMP.jl/stable/packages/Gurobi/). A license must support the installed solver version and models of this size; the limited-size license is insufficient for the 100-by-100 example.

In [ ]:
using JuMP, Gurobi, HiGHS
using Random, Statistics, LinearAlgebra

# Reuse one environment for all Gurobi models; it finds the default license.
gurobi_env = Gurobi.Env(; output_flag = 0)
gurobi_optimizer = () -> Gurobi.Optimizer(gurobi_env)

license_check = Model(gurobi_optimizer)
set_silent(license_check)
@variable(license_check, 0 <= license_x <= 1)
@objective(license_check, Max, license_x)
optimize!(license_check)
if !is_solved_and_feasible(license_check; allow_local = false)
    error("Gurobi license check failed: $(termination_status(license_check))")
end
println("Gurobi is ready; test objective = ", objective_value(license_check))

## Build and solve the location model

There are 100 candidate facilities and 100 customers. Opening facility $i$ costs $f_i=0.5$, and assigning customer $j$ to facility $i$ costs $c_{ij}$, drawn uniformly from $[0,1)$. The fixed seed makes the instance repeatable. No external data files are needed.

Let $z_i=1$ if facility $i$ opens, and $y_{ij}=1$ if it serves customer $j$. Minimize opening and assignment costs:

$$
\begin{aligned}
\min\quad &\sum_i f_i z_i+\sum_{i,j}c_{ij}y_{ij}\\
\text{s.t.}\quad &\sum_i y_{ij}=1 &&\text{for each customer }j,\\
&y_{ij}\le z_i &&\text{for each facility }i\text{ and customer }j,\\
&z_i,y_{ij}\in\{0,1\}.
\end{aligned}
$$

With binary $z$, the aggregate linking constraint $\sum_j y_{ij}\le n_Jz_i$ gives the same integer feasible solutions as the individual constraints $y_{ij}\le z_i$. The individual constraints give a stronger LP relaxation. The code uses the individual constraints and retains the aggregate version as a commented alternative.

In [ ]:
Random.seed!(79883274598)

nI = 100  # number of facilities
nJ = 100  # number of customers

# generate random problem instance
f = 0.5*ones(nI)
c = rand(nI,nJ)

m = Model(gurobi_optimizer)
set_silent(m)

@variable(m, z[1:nI], Bin)         # binary constraint
# @variable(m, 0 <= z[1:nI] <= 1)    # LP relaxation

@variable(m, y[1:nI,1:nJ], Bin)     # binary constraint
#@variable(m, y[1:nI,1:nJ] >= 0)   # LP relaxation

@constraint(m, cc[j = 1:nJ], sum( y[i,j] for i=1:nI ) == 1)

# These constraints are equivalent when z is binary.
# @constraint(m, cr[i=1:nI], sum( y[i,j] for j=1:nJ ) <= nJ*z[i])
@constraint(m, cr[i=1:nI,j=1:nJ], y[i,j] <= z[i])

@objective(m, Min, dot(f,z) + dot(c,y))

@time(optimize!(m))
if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
end

println(value.(z))
zopt=value.(z)
println("total number of facilities is ",sum(zopt))
println("total cost is ", objective_value(m))

## Optional: compare HiGHS and Gurobi solve times

Set `run_benchmarks = true` below and run the remaining cells to compare both solvers on the same seeded instances, with 5 through 50 facilities and customers and 10 trials per size. This performs 200 solves and can take several minutes. Leave it `false` for a first **Run All**.

The timing models use binary facility variables `x` and continuous nonnegative assignment variables `y`. For any fixed set of open facilities, each customer can be assigned completely to a cheapest open facility, so an optimal integral assignment exists without declaring `y` binary. Both solvers receive the same formulation and data.

Timings measure `optimize!`, including any compilation during the call, and vary with the computer and solver versions. The plot illustrates this experiment; it does not establish a general ranking of the solvers.

In [ ]:
run_benchmarks = false
nvals = [5,10,15,20,25,30,35,40,45,50]
ntrials = 10
tt_highs = zeros(length(nvals), ntrials)
tt_gurobi = zeros(length(nvals), ntrials);

In [ ]:
if run_benchmarks
    let t = tt_highs
        for count in 1:ntrials
            println("HiGHS trial ", count)
            for (k,n) in enumerate(nvals)
                nI = n
                nJ = n
                Random.seed!(count)
                f = 0.5*ones(nI)
                c = rand(nI,nJ)

                m = Model(HiGHS.Optimizer)
                set_silent(m)
                @variable(m, x[1:nI], Bin)
                @variable(m, y[1:nI,1:nJ] >= 0)
                @constraint(m, cc[j=1:nJ], sum(y[i,j] for i=1:nI) == 1)
                @constraint(m, cr[i=1:nI,j=1:nJ], y[i,j] <= x[i])
                @objective(m, Min, dot(f,x) + dot(vec(c),vec(y)))
                t[k,count] = @elapsed optimize!(m)
                if !is_solved_and_feasible(m; allow_local = false)
                    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
                end
            end
        end
    end
end

In [ ]:
if run_benchmarks
    let t = tt_gurobi
        for count in 1:ntrials
            println("Gurobi trial ", count)
            for (k,n) in enumerate(nvals)
                nI = n
                nJ = n
                Random.seed!(count)
                f = 0.5*ones(nI)
                c = rand(nI,nJ)

                m = Model(gurobi_optimizer)
                set_silent(m)
                @variable(m, x[1:nI], Bin)
                @variable(m, y[1:nI,1:nJ] >= 0)
                @constraint(m, cc[j=1:nJ], sum(y[i,j] for i=1:nI) == 1)
                @constraint(m, cr[i=1:nI,j=1:nJ], y[i,j] <= x[i])
                @objective(m, Min, dot(f,x) + dot(vec(c),vec(y)))
                t[k,count] = @elapsed optimize!(m)
                if !is_solved_and_feasible(m; allow_local = false)
                    error("No optimal primal solution: $(termination_status(m)), $(primal_status(m))")
                end
            end
        end
    end
end

In [ ]:
if run_benchmarks
    tt_highs_mean = vec(mean(tt_highs; dims = 2))
    tt_gurobi_mean = vec(mean(tt_gurobi; dims = 2))
    println("Gurobi mean seconds: ", tt_gurobi_mean)
    println("HiGHS mean seconds: ", tt_highs_mean)
else
    println("Timing comparison skipped. Set run_benchmarks = true to enable it.")
end

In [ ]:
using Plots

if run_benchmarks
    comparison_plot = plot(nvals, tt_highs_mean;
        yscale = :log10,
        label = "HiGHS",
        lw = 2,
        xlabel = "number of facilities and customers",
        ylabel = "time to solve (seconds)",
        title = "comparison of different solvers",
        legend = :best,
        size = (1000, 400),
    )
    plot!(comparison_plot, nvals, tt_gurobi_mean; label = "Gurobi", lw = 2)
    display(comparison_plot)
end